In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import folium
import joblib
import os

df = pd.read_csv(os.path.join('..', 'data', 'processed', 'features.csv'))
model = joblib.load(os.path.join('..', 'models', 'xgb_best.pkl'))

print(df.shape)

(4548, 141)


In [2]:
drop_cols = ['EventID', 'GaugeID', 'Start Date', 'End Date',
             'Peak FL Date', 'Flood Type', 'Peak Discharge Date',
             'Station', 'River Name/ Tributory/ SubTributory',
             'Basin', 'Privacy', 'Reliability',
             'KoppenGeiger Climate Type', 'Land cover',
             'Soil type', 'lithology type', 'State']

df_model = df.drop(columns=[c for c in drop_cols if c in df.columns])
obj_cols  = df_model.select_dtypes(include=['str', 'datetime64']).columns.tolist()
df_model  = df_model.drop(columns=obj_cols)

X = df_model.drop(columns=['label'])

# probability of severe flood for each event
df['severe_prob'] = model.predict_proba(X)[:, 1]

print(df[['GaugeID', 'severe_prob']].head())

                 GaugeID  severe_prob
0  INDOFLOODS-gauge-1010     0.003513
1  INDOFLOODS-gauge-1010     0.013621
2  INDOFLOODS-gauge-1010     0.002861
3  INDOFLOODS-gauge-1010     0.988768
4  INDOFLOODS-gauge-1012     0.013050


C:\Users\USER\AppData\Local\Temp\ipykernel_10032\4206889034.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['severe_prob'] = model.predict_proba(X)[:, 1]


In [3]:
# mean severe flood probability per gauge across all its events
gauge_risk = df.groupby('GaugeID').agg(
    mean_severe_prob = ('severe_prob', 'mean'),
    event_count      = ('EventID', 'count'),
    lat              = ('Latitude', 'mean'),
    lon              = ('Longitude', 'mean'),
).reset_index()

print(gauge_risk.shape)
print(gauge_risk.describe().round(3))

(155, 5)
       mean_severe_prob  event_count      lat      lon
count           155.000      155.000  155.000  155.000
mean              0.435       29.342   16.843   78.567
std               0.302       55.000    5.128    4.802
min               0.000        1.000    8.160   72.792
25%               0.173        2.000   11.712   75.586
50%               0.396       10.000   17.254   76.891
75%               0.665       28.500   20.903   79.514
max               0.999      419.000   26.729   91.592


In [4]:
# bin continuous probability into 4 risk tiers
gauge_risk['risk_category'] = pd.cut(
    gauge_risk['mean_severe_prob'],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)

print(gauge_risk['risk_category'].value_counts())

risk_category
Low          52
Medium       40
Very High    32
High         31
Name: count, dtype: int64


In [5]:
color_map = {
    'Low'      : 'green',
    'Medium'   : 'orange',
    'High'     : 'red',
    'Very High': 'darkred'
}

# centre map on mean coordinates of all gauges
m = folium.Map(location=[gauge_risk['lat'].mean(),
                          gauge_risk['lon'].mean()],
               zoom_start=5)

for _, row in gauge_risk.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color=color_map[row['risk_category']],
        fill=True,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"<b>{row['GaugeID']}</b><br>"
            f"Risk: {row['risk_category']}<br>"
            f"Severe prob: {row['mean_severe_prob']:.2f}<br>"
            f"Events: {int(row['event_count'])}",
            max_width=200
        )
    ).add_to(m)

# save as interactive HTML
os.makedirs(os.path.join('..', 'outputs', 'maps'), exist_ok=True)
map_path = os.path.join('..', 'outputs', 'maps', 'flood_risk_map.html')
m.save(map_path)
print(f"Map saved to {map_path}")

Map saved to ..\outputs\maps\flood_risk_map.html


In [6]:
gauge_risk.to_csv(
    os.path.join('..', 'outputs', 'gauge_risk_scores.csv'), index=False)
print("Risk scores saved.")
print(gauge_risk[['GaugeID','mean_severe_prob','risk_category','event_count']]\
      .sort_values('mean_severe_prob', ascending=False).head(10))

Risk scores saved.
                   GaugeID  mean_severe_prob risk_category  event_count
109   INDOFLOODS-gauge-679          0.999481     Very High           70
10   INDOFLOODS-gauge-1076          0.998762     Very High           10
77    INDOFLOODS-gauge-446          0.993514     Very High            2
34    INDOFLOODS-gauge-345          0.993381     Very High            1
9    INDOFLOODS-gauge-1074          0.993298     Very High            1
144   INDOFLOODS-gauge-898          0.992948     Very High            3
86    INDOFLOODS-gauge-585          0.988657     Very High            2
54    INDOFLOODS-gauge-404          0.988019     Very High            1
52    INDOFLOODS-gauge-399          0.987741     Very High            1
72    INDOFLOODS-gauge-434          0.977605     Very High            1


In [7]:
# mark gauges with fewer than 5 events as low confidence
gauge_risk['confidence'] = gauge_risk['event_count'].apply(
    lambda x: 'High' if x >= 5 else 'Low'
)

print("Risk category vs confidence:")
print(pd.crosstab(gauge_risk['risk_category'], gauge_risk['confidence']))

# save updated table
gauge_risk.to_csv(
    os.path.join('..', 'outputs', 'gauge_risk_scores.csv'), index=False)
print("\nUpdated risk scores saved.")

Risk category vs confidence:
confidence     High  Low
risk_category           
Low              37   15
Medium           31    9
High             19   12
Very High        12   20

Updated risk scores saved.


In [8]:
honest_model = joblib.load(os.path.join('..', 'models', 'xgb_honest_final.pkl'))
top_features = pd.read_csv(
    os.path.join('..', 'data', 'processed', 'top_features.csv')
)['0'].tolist()

print(f"Honest model loaded. Features: {len(top_features)}")

Honest model loaded. Features: 40


In [9]:
df = pd.read_csv(os.path.join('..', 'data', 'processed', 'features.csv'))

drop_cols = ['EventID', 'GaugeID', 'Start Date', 'End Date',
             'Peak FL Date', 'Flood Type', 'Peak Discharge Date',
             'Station', 'River Name/ Tributory/ SubTributory',
             'Basin', 'Privacy', 'Reliability',
             'KoppenGeiger Climate Type', 'Land cover',
             'Soil type', 'lithology type', 'State',
             'Peak Flood Level (m)', 'Peak Discharge Q (cumec)',
             'Flood Volume (cumec)', 'Event Duration (days)',
             'Recession Time (day)', 'Time to Peak (days)',
             'Num Peak FL']

df_model = df.drop(columns=[c for c in drop_cols if c in df.columns])
obj_cols  = df_model.select_dtypes(include=['str','datetime64']).columns.tolist()
df_model  = df_model.drop(columns=obj_cols)

X = df_model.drop(columns=['label'])[top_features]

df['severe_prob_honest'] = honest_model.predict_proba(X)[:, 1]

print(df[['GaugeID', 'severe_prob_honest']].head())

                 GaugeID  severe_prob_honest
0  INDOFLOODS-gauge-1010            0.625342
1  INDOFLOODS-gauge-1010            0.614298
2  INDOFLOODS-gauge-1010            0.643246
3  INDOFLOODS-gauge-1010            0.692674
4  INDOFLOODS-gauge-1012            0.396519


C:\Users\USER\AppData\Local\Temp\ipykernel_10032\1433771528.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['severe_prob_honest'] = honest_model.predict_proba(X)[:, 1]


In [10]:
gauge_risk_honest = df.groupby('GaugeID').agg(
    mean_severe_prob = ('severe_prob_honest', 'mean'),
    event_count      = ('EventID', 'count'),
    lat              = ('Latitude', 'mean'),
    lon              = ('Longitude', 'mean'),
).reset_index()

gauge_risk_honest['risk_category'] = pd.cut(
    gauge_risk_honest['mean_severe_prob'],
    bins=[0, 0.25, 0.5, 0.75, 1.0],
    labels=['Low', 'Medium', 'High', 'Very High']
)

gauge_risk_honest['confidence'] = gauge_risk_honest['event_count'].apply(
    lambda x: 'High' if x >= 5 else 'Low')

print(gauge_risk_honest['risk_category'].value_counts())
print(f"\nHigh confidence gauges: {(gauge_risk_honest['confidence']=='High').sum()}")

risk_category
High         71
Medium       59
Very High    13
Low          12
Name: count, dtype: int64

High confidence gauges: 99


In [12]:
color_map = {
    'Low'      : 'green',
    'Medium'   : 'orange',
    'High'     : 'red',
    'Very High': 'darkred'
}

m2 = folium.Map(location=[gauge_risk_honest['lat'].mean(),
                           gauge_risk_honest['lon'].mean()],
                zoom_start=5,
                tiles='CartoDB positron')

for _, row in gauge_risk_honest.iterrows():
    # dashed border for low confidence gauges
    weight     = 1 if row['confidence'] == 'High' else 3
    dash_array = None if row['confidence'] == 'High' else '5 5'

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=7 if row['confidence'] == 'High' else 5,
        color=color_map[row['risk_category']],
        fill=True,
        fill_opacity=0.8 if row['confidence'] == 'High' else 0.4,
        weight=weight,
        dash_array=dash_array,
        popup=folium.Popup(
            f"<b>{row['GaugeID']}</b><br>"
            f"Risk: {row['risk_category']}<br>"
            f"Severe prob: {row['mean_severe_prob']:.2f}<br>"
            f"Events: {int(row['event_count'])}<br>"
            f"Confidence: {row['confidence']}",
            max_width=200
        )
    ).add_to(m2)

# legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px;border-radius:8px;font-size:13px">
  <b>Flood Severity Risk</b><br>
  <i style="color:darkred">&#9679;</i> Very High<br>
  <i style="color:red">&#9679;</i> High<br>
  <i style="color:orange">&#9679;</i> Medium<br>
  <i style="color:green">&#9679;</i> Low<br>
  <hr style="margin:4px 0">
  <i>Faded = low confidence</i>
</div>"""
m2.get_root().html.add_child(folium.Element(legend_html))

map_path2 = os.path.join('..', 'outputs', 'maps', 'flood_risk_map_honest.html')
m2.save(map_path2)

gauge_risk_honest.to_csv(
    os.path.join('..', 'outputs', 'gauge_risk_honest.csv'), index=False)

print(f"Honest risk map saved.")

Honest risk map saved.
